# V2 视觉材料子流程

知识框架内可独立执行的分支；基础材料准备流程直接调用下方同一个图片审核／发布子图。

输入为固定 Lance DatasetRef，每行 concept＋images。只读取原始图及来源，不把旧 keep 或文章通过当成新视觉发布依据。

## 标注字段阅读指南

输出的 `visual_image_meta` Lance 阶段表 按图片保存三个层次的信息：

| 层次 / 字段 | 中文含义 | 如何理解 |
| --- | --- | --- |
| 图片索引 `image_metadata.description` | `caption` 画面描述；`representation` 表现形式；`view_tags` 视角；`objects` 可见对象及位置/特征；`text_regions` 图中文字；`observability_issues` 看图障碍；`uncertainties` 无法确认的要点 | 机器检索描述，单独不能认证概念身份或知识事实 |
| 概念审核 `concept_review` | `decision` 保留 keep / 排除 exclude / 待定 pending；`concept_relation` 与概念的具体关系；`reason` 理由；`visible_information` 可见信息；`limitations` 限制 | `concept_relation=target` 指概念本身，不是训练目标图；`protocol_valid` 只说明字段符合协议 |
| 发布支持 `visual_support` | `supports` 能支持什么；`region` 对应区域；`limitations` 支持边界 | 顶层值来自已发布材料；未发布则为 null，内部候选描述不能替代发布结果 |
| 发布状态 `publication_status` | reviewed 已机器审核发布 / not_published 未发布 | 与图片索引状态 machine_index_only 不冲突；human_reviewed=false 表示尚未人工认证 |
| 任务角色 `task_roles` | 具体题目中的参考图/目标图分工 | 本轮 null 表示尚未分配；仍需任务级适配及答案泄露检查 |
| 分辨率 `resolution` | width / height 原图按方向处理后的像素宽高；stored_width / stored_height 编码尺寸；megapixels 百万像素；aspect_ratio 宽高比 | 与送入模型的缩放图片、notebook 显示尺寸区分；分辨率不是清晰度评分 |
| 文件与溯源 | byte_size 字节数；format 编码格式；sha256 原图指纹；source 来源；image_metadata.provenance 标注来源；review_calls 调用记录 | 复用旧预标注与本轮补充标注可追踪；verified_bytes 只认证字节校验 |

`null` 是未提供/未赋值，`[]` 是未记录条目，不能一概解释为失败或问题不存在。审核使用限制时，要区分“这张图能看见什么”与“它足以回答哪道题”。


## 当前材料表怎么读

原始图片来源是 `raw/images.lance`，一行一个 SHA；文档来源是 `raw/documents.lance`，一行一个确定版本。每种实体固定一个 DatasetRef，分片由 Lance 内部管理。

| 字段 | 含义 |
| --- | --- |
| `concepts` | 来源关联的概念去重列表，便于筛选；不等于审核通过 |
| `sources` | 每条采集来源的概念、查询词、URL、图注、版权与时间；不同来源不会相互覆盖 |
| `availability` | 图片已有 Lance Blob 或只有 metadata；没有字节会明确报告 |
| `resolution` | 实测显示宽高、存储宽高、MP、宽高比和测量来源；null 表示未测量 |
| `sources[].declared_width/height` | 来源声明的尺寸，不能当作实测尺寸 |
| `document_id / content_sha256` | 文档版本身份 / 正文摘要；不同版本独立保留 |
| `text / sections / images` | 正文或 typed 章节、文内图片引用，不把正文和图片 Blob 藏进来源 JSON |

图片客观索引、概念支持审核、任务参考/目标角色仍是三种不同结论，分别保留协议和引用。文章交付携带 `published_passages` 和 `published_images`，后者是最终配图 ID 到原图 SHA/来源的精确绑定；`images` 是原始候选池，不能因在池中就成为答案参考。

当前发布：原始材料 `material_entities_20260921`、知识文章 `knowledge_articles_20260921`、视觉材料 `visual_images_20260921`。分辨率未全库补跑；实际用图时测量。

原始图片与策展图片是两张表：`raw/images.lance` 保存字节和采集来源，`curated/images.lance` 保存标签与审核。`source_refs[]` 表示处理所用的固定原始 DatasetRef；补标签只更新 curated，采集更新 raw 不会自动改变已有发布。复用描述需要单独传入 curated 的固定引用。


In [ ]:
from pathlib import Path
import sys
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))


In [ ]:
from curation.preparation.stages import EncodeStage, stage_uri, read_stage, from_stage_row, PIPELINE_STAGE_ROWS
from pathlib import Path
from demiflow.standalone import local_data
from curation.preparation.contracts import ROOT, run_lock
from pathlib import Path
from curation.preparation.config import DEFAULT
from curation.preparation.ops.image_filter import IMAGE_FILTER_DEFAULTS, RecordPrimaryImageSelection, PrepareImageReview, ApplyConfirmedImageSelection
from curation.preparation.ops.image_selection import SelectAvailableImages, BatchImageSelection, merge_image_decisions, SelectRelatedMaterials
from curation.preparation.ops.visual_materials import PublishVisualMaterials, ExportVisualMetadata
from curation.preparation.ops.visual_inputs import PrepareVisualInput, visual_pack, merge_visual_packs
from curation.preparation.image_filter_runtime import image_prompt_data, review_needed, save_image_filter_policy
from curation.preparation.local_review_service import image_review_service
from curation.preparation.visual_pipeline import freeze_visual_run
from project import resolve_root as DATASET_ROOT


## 1．配置

默认不执行。prepare 检查字节、复用索引并冻结中立像素请求，零模型调用；review 执行双模型审核；export 发布独立视觉材料。配置、代码或输入改变用新 run。同版本可从 prepare 继续。

In [ ]:
INPUT = None  # 填写包含 concept/images 的固定 DatasetRef；历史文件先由迁移工具入湖
RUN = ROOT/'curation/preparation/runs/pipeline_v2_visual_review'
DATASET = DATASET_ROOT()
CONFIG = {'max_calls': 16}  # 大批运行应显式配置预算、模型与服务策略。
THROUGH = 'prepare'
EXECUTE = False


## 2．共享图片审核与发布编排

实际算子：BatchImageSelection → Qwen → PrepareImageReview → Gemma → ApplyConfirmedImageSelection；审核分歧保持 pending。PublishVisualMaterials 是单独发布步骤。以下也是基础材料准备流程调用的实现。

In [ ]:
from curation.preparation.stages import EncodeStage, stage_uri, read_stage, from_stage_row, PIPELINE_STAGE_ROWS
def run_image_review(blocks, run, config, version, *, through='review'):
    """Shared graph: callers own the run lock and immutable input/config version."""
    run = Path(run)
    tables = run / 'datasets'
    image_requests = blocks.flat_map(BatchImageSelection(config.get('image_batch_size', 4), config['image_identity_definitions'], neutral=True, visual_publication=True)).map(EncodeStage('image_requests')).checkpoint_lance(stage_uri(run, 'image_requests'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'image_requests').map(from_stage_row)
    if through == 'prepare':
        return image_requests
    primary_data = image_prompt_data(run, config)
    primary = read_stage(run, 'image_requests', data=primary_data).map_prompt_async('select_images', config='knowledge.yaml', inputs={'payload': 'image_prompt', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=config.get('image_primary_concurrency', 1), queue_depth=2 * config.get('image_primary_concurrency', 1)).map(RecordPrimaryImageSelection()).map(EncodeStage('image_primary')).checkpoint_lance(stage_uri(run, 'image_primary'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'image_primary').map(from_stage_row)
    review_rows = primary.map(PrepareImageReview()).map(EncodeStage('image_review_inputs')).checkpoint_lance(stage_uri(run, 'image_review_inputs'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'image_review_inputs').map(from_stage_row)
    review_data = image_prompt_data(run, config, review=True)
    review_requests = read_stage(run, 'image_review_inputs', data=review_data).filter(lambda r: r['review_required'])
    review_path = stage_uri(run, 'image_review_responses')
    with image_review_service(run, config, needed=review_needed(review_requests, review_path, version + ':image_review_responses')):
        reviewed = review_requests.map_prompt_async('select_images', config='knowledge.yaml', inputs={'payload': 'image_prompt', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=config['image_review_concurrency'], queue_depth=config['image_review_concurrency']).map(EncodeStage(Path(review_path).stem)).checkpoint_lance(stage_uri(run, Path(review_path).stem), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + Path(review_path).stem).map(from_stage_row)
    image_decisions = reviewed.union(review_rows.filter(lambda r: not r['review_required'])).map(ApplyConfirmedImageSelection()).map(EncodeStage('image_relevance')).checkpoint_lance(stage_uri(run, 'image_relevance'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'image_relevance').map(from_stage_row).reduce_by_key('case_id', merge_image_decisions)
    save_image_filter_policy(run, config)
    return image_decisions

def publish_visual_branch(related, run, version):
    published = related.map(PublishVisualMaterials()).map(EncodeStage('visual_materials')).checkpoint_lance(stage_uri(run, 'visual_materials'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'visual_materials').map(from_stage_row)
    published.flat_map(ExportVisualMetadata()).map(EncodeStage('visual_image_meta')).checkpoint_lance(stage_uri(run, 'visual_image_meta'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'visual_image_meta').map(from_stage_row)
    from curation.preparation.publication import publish_entity_stage
    publish_entity_stage(run, 'visual')
    return published


## 3．独立入口编排

原始图片检查与索引复用 → 共享双模型审核 → 独立发布。无需执行正文清洗、联合提炼或文章终审。输出 `knowledge_base` Lance 阶段表 沿用下游发布接口，publication_kind=visual_materials、knowledge=[]，不冒充知识文章。

In [ ]:
from curation.preparation.stages import EncodeStage, stage_uri, read_stage, from_stage_row, PIPELINE_STAGE_ROWS
def run_visual_pipeline(run, input_path, dataset=None, *, through='prepare', model_config=None, project=ROOT):
    """Independent knowledge subflow: no document parsing or article model calls."""
    if through not in {'prepare', 'review', 'export'}:
        raise ValueError('Expected prepare, review or export')
    run, dataset = Path(run).resolve(), Path(dataset or DATASET_ROOT()).resolve()
    from demiflow.lance.refs import DatasetRef
    input_ref = DatasetRef.from_dict(input_path)
    config = {**DEFAULT, **IMAGE_FILTER_DEFAULTS, **(model_config or {})}
    config.setdefault('image_annotations_ref', None)
    from curation.preparation.config import check_run_location
    check_run_location(run, project, dataset)
    with run_lock(run):
        version, source = freeze_visual_run(run, input_path, dataset, config, Path(project))
        data = local_data()
        from curation.preparation.ops.visual_inputs import read_visual_input
        blocks = read_visual_input(data, input_ref, dataset, config.get('concepts')).map(PrepareVisualInput(run, dataset, source, config.get('image_annotations_ref'), config.get('image_annotation_config_id'))).map(SelectAvailableImages()).map(EncodeStage('visual_inputs')).checkpoint_lance(stage_uri(run, 'visual_inputs'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'visual_inputs').map(from_stage_row)
        decisions = run_image_review(blocks, run, config, version, through=through)
        if through == 'prepare':
            return decisions
        related = blocks.join(decisions, on='case_id', how='left').map(SelectRelatedMaterials()).map(EncodeStage('visual_reviewed')).checkpoint_lance(stage_uri(run, 'visual_reviewed'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'visual_reviewed').map(from_stage_row)
        if through == 'review':
            return related
        published = publish_visual_branch(related, run, version)
        return published.map(visual_pack).reduce_by_key('concept', merge_visual_packs).map(EncodeStage('knowledge_base')).checkpoint_lance(stage_uri(run, 'knowledge_base'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'knowledge_base').map(from_stage_row)


## 4．执行与只读检查

In [ ]:
if EXECUTE:
    result = run_visual_pipeline(RUN, INPUT, DATASET, through=THROUGH, model_config=CONFIG)
    print('records:', sum(1 for _ in result.iter_rows()))
else:
    print('未执行。确认 INPUT、RUN 和预算后，设置 EXECUTE=True。')


In [ ]:
from curation.preparation.stages import stage_uri, read_stage
if Path(stage_uri(RUN, 'knowledge_base')).exists():
    display(read_stage(RUN, 'knowledge_base').take(3))
else:
    print('尚未发布；运行结果存于 Lance。')


## 5．已有响应的程序复验
只用于程序校验修复：检查原提示词、模型、像素和 checkpoint 后，在新 run 重验已保存响应，不调用模型、不补造缺失审核。原始结果保留。


In [ ]:
from curation.preparation.stages import EncodeStage, stage_uri, read_stage, from_stage_row, PIPELINE_STAGE_ROWS
def replay_visual_pipeline(run, parent, *, project=ROOT):
    """Replay complete recorded model responses after a parser fix, with zero new calls."""
    from curation.preparation.visual_pipeline import freeze_visual_replay
    from curation.preparation.ops.visual_inputs import ReplayVisualResponse
    from curation.preparation.config import check_run_location
    run, parent = (Path(run).resolve(), Path(parent).resolve())
    check_run_location(run, project, Path(project) / 'datasets')
    if run == parent:
        raise ValueError('Replay must use a new run')
    with run_lock(run):
        version = freeze_visual_replay(run, parent, project)
        data = local_data()
        blocks = read_stage(parent, 'visual_inputs', data=data)
        responses = read_stage(parent, 'image_review_responses', data=data).take_all()
        decisions = read_stage(parent, 'image_primary', data=data).map(ReplayVisualResponse(responses)).map(EncodeStage('image_relevance')).checkpoint_lance(stage_uri(run, 'image_relevance'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'image_relevance').map(from_stage_row).reduce_by_key('case_id', merge_image_decisions)
        related = blocks.join(decisions, on='case_id', how='left').map(SelectRelatedMaterials()).map(EncodeStage('visual_reviewed')).checkpoint_lance(stage_uri(run, 'visual_reviewed'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'visual_reviewed').map(from_stage_row)
        published = publish_visual_branch(related, run, version)
        return published.map(visual_pack).reduce_by_key('concept', merge_visual_packs).map(EncodeStage('knowledge_base')).checkpoint_lance(stage_uri(run, 'knowledge_base'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'knowledge_base').map(from_stage_row)
